# 기초데이터과학 (01분반)

## 13-1. 데이터 분석 케이스 스터디 2
### 주택 가격: 중개사 설명 검토하기

#### 데이터 분석 단계 (12주차와 같음)
| 단계 | 할 일 | 이번 시간에 하는 일 |
|---|---|---|
| 1. 질문 | 무엇을 알아내려는지 한 문장으로 정함 | 중개사의 설명이 데이터로 뒷받침되는지 묻는 질문을 정함 |
| 2. 데이터 확인 | 크기, 변수, 자료형, 결측치 확인 | 중복 행, 가격 분포 확인 |
| 3. 정제 | 결측치, 이상치, 존재할 수 없는 값을 판단하고 처리 | 중복 행, 정상 거래로 보기 어려운 행 제외 |
| 4. 탐색 | 집계와 그래프로 패턴 확인 | **주장의 숫자 직접 계산하기**<br>**다른 이유 찾기**: 다른 요인 후보 떠올리기<br>**확인하기**: 후보 요인이 비슷한 집끼리 나누어 비교 |
| 5. 근거 | 상관 분석, 가설 검정으로 판단 | **확인하기**: 나누어 비교한 차이를 t 검정으로 확인 |
| 6. 결론과 한계 | 알게 된 것과 이 분석으로 말할 수 없는 것을 정리 | **판단하기**: 중개사의 설명을 받아들일지 정하고 한계 정리 |

#### 지난주와 달라지는 점
- 12주차에는 팀장이 정한 질문에 답했다면, 이번에는 다른 사람의 **주장을 검토**함
- 단계 4 ~ 6에서 차이를 만든 다른 요인 후보를 스스로 떠올리고, 확인한 결과를 보고 스스로 판단함
- 이번 시간의 핵심은 차이를 만든 **다른 요인이 숨어 있는지** 찾아보는 것임
  - **교란 변수** (11주차): 두 변수 모두에 영향을 주어, 두 변수가 서로 관련 있는 것처럼 보이게 만드는 다른 변수

---

## 단계 1. 질문

### 상황

- 여러분의 가족이 미국 뉴욕주 사라토가 카운티에서 집을 사려고 합니다.
- 부동산 중개사가 매물을 소개하면서 다음과 같이 설명했습니다.

> 1. "벽난로가 있는 집은 **6.6만 달러** 더 비쌉니다."
> 2. "중앙냉방이 있는 집은 **6.8만 달러** 더 비쌉니다."
> 3. "침실이 4개인 집은 3개인 집보다 **6.5만 달러** 더 비쌉니다."

- 가족이 여러분에게 다음과 같이 부탁했습니다.

> "중개사 말대로 그만큼 더 내는 게 맞는지 모르겠어.
> 지난 거래 기록으로 확인해 줄 수 있니?"

### 이번 시간의 질문

> **중개사의 설명대로 그만큼 더 내는 것이 합리적인가?**

- 확인할 것: 설명에 나온 가격 차이가 정말 벽난로, 중앙냉방, 침실 수 때문인지, 아니면 교란 변수 때문인지
- 이번 강의에서는 **1번 설명**을 함께 검토하고, 실습에서 2번과 3번을 직접 검토함

---

## 단계 2. 데이터 확인

### 데이터
- 2006년 미국 뉴욕주 사라토가 카운티에서 거래된 주택 1728채의 가격과 특성
- 출처: R 패키지 mosaicData의 `SaratogaHouses` (Richard De Veaux 수집)
  - https://vincentarelbundock.github.io/Rdatasets/doc/mosaicData/SaratogaHouses.html
- 미국 자료이므로 가격은 달러, 면적은 제곱피트, 대지는 에이커 단위임

| 주요 변수 | 의미 |
|---|---|
| `price` | 거래 가격 (달러) |
| `livingArea` | 거주 면적 (제곱피트) |
| `lotSize` | 대지 면적 (에이커) |
| `landValue` | 토지 가치 (달러) |
| `age` | 집 나이 (년) |
| `pctCollege` | 동네 주민 중 대학 졸업자 비율 (%) |
| `bedrooms`, `bathrooms`, `rooms` | 침실 수, 욕실 수, 방 수 (욕실 0.5는 샤워 시설이 없는 욕실) |
| `fireplaces` | 벽난로 개수 |
| `heating`, `fuel` | 난방 방식, 난방 연료 |
| `sewer` | 하수 처리 방식 |
| `waterfront` | 물가에 있는 집인지 (Yes, No) |
| `newConstruction` | 새로 지은 집인지 (Yes, No) |
| `centralAir` | 중앙냉방이 있는지 (Yes, No) |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

houses = pd.read_csv('saratoga_houses.csv')
print(houses.head())

In [ ]:
print(houses.shape)
print()
print(houses.isna().sum().sum(), "개의 결측치")
print(houses.duplicated().sum(), "개의 중복 행")
print()
print(houses['price'].describe().round(0))

- 1728 행. 한 행이 집 한 채의 거래를 나타냄
- 결측치는 없음
- 모든 값이 같은 **중복 행**이 7개 있음
- 가장 낮은 가격이 5000 달러임. 집 한 채의 가격으로 가능한 값일까?

---

## 단계 3. 정제 — 판단이 필요한 값

### 중복 행

#### duplicated() 함수 사용 예제
- 각 행이 앞에서 이미 나온 행과 동일한지 True/False로 알려주는 함수

In [ ]:
df = pd.DataFrame({'name':  ['민수', '지영', '민수', '현우', '민수'],
                   'city':  ['서울', '부산', '서울', '대전', '부산'],
                   'score': [80, 90, 80, 70, 85]})
print(df)
print()
print("keep: first")
print(df.duplicated()) # 처음 나온 행은 False, 그 뒤에 나온 같은 행만 True
print()
print("중복 개수", df.duplicated().sum())
print()

# keep 매개변수 : 어느 행을 원래 행으로 볼지 중복 행으로 볼지 정함 - 'first' (기본값), 'last', False
print("keep: last")
print(df.duplicated(keep='last')) # 마지막에 나온 행은 False, 그 앞의 같은 행이 True
print()
print("keep: False")
print(df.duplicated(keep=False)) # 중복된 행을 모두 True

#### 주택 데이터의 중복 행 확인

In [ ]:
# keep=False : 중복된 행을 모두 표시 (첫 번째 행도 포함)
dup = houses[houses.duplicated(keep=False)]
print(dup.sort_values('price')[['price', 'livingArea', 'age', 'lotSize', 'landValue']])

#### 판단해 볼 것
- 16개 열의 값이 모두 같은 집이 7쌍 있음
- 이 행들을 그대로 두어도 될까?

> 나의 판단과 이유:

<details>
<summary><b>판단의 근거 보기</b> (클릭)</summary>

- 가격, 면적, 집 나이, 대지, 토지 가치까지 모두 같은 집이 따로 거래되었을 가능성은 낮음
- 행 번호를 보면 대부분 45행씩 떨어져 있음. 자료의 한 부분이 **두 번 입력**된 것으로 보임
- 같은 기록이 두 번 들어간 것으로 판단하여 **하나만 남김**
- 7행은 전체의 0.4%라서 결론에 주는 영향은 작지만, 판단한 내용은 기록해 둠

</details>

In [ ]:
houses = houses.drop_duplicates()
print(houses.shape)

### 가격이 매우 낮은 집

In [ ]:
print(houses.sort_values('price')[['price', 'landValue', 'livingArea', 'age']].head())

#### 판단해 볼 것
- 가장 싼 집들의 가격(`price`)과 토지 가치(`landValue`)를 비교해 볼 것
- 이 행들을 그대로 두고 분석해도 될까?

> 나의 판단과 이유:

<details>
<summary><b>판단의 근거 보기</b> (클릭)</summary>

- 가장 싼 집 3채는 **가격이 토지 가치보다 낮음** (5000 달러에 팔린 집의 토지 가치는 35800 달러)
- 집값은 땅의 가치와 건물의 가치를 합친 것인데, 땅의 가치보다도 낮게 팔렸다는 뜻이므로 가족 간 거래처럼 정상적인 거래가 아닐 가능성이 큼
- 이번 분석은 시장에서 거래되는 집값을 다루므로 이 행들은 목적에 맞지 않음
- 가격이 토지 가치보다 낮은 행을 **분석에서 제외**함
- 20000 달러, 25000 달러인 집은 토지 가치보다 비싸게 팔렸으므로 남겨 둠
- 제외 기준을 논리적인 **규칙**으로 정해야 다른 사람도 같은 판단을 할 수 있음

</details>

In [ ]:
houses = houses.query('price >= landValue')
print(houses.shape)

### 단위 바꾸기
- 제곱피트와 달러는 우리에게 익숙하지 않으므로 비교하기 쉬운 단위의 열을 추가함
  - 거주 면적: 1 제곱피트 = 0.0929 제곱미터
  - 가격: 만 달러 단위 (예: 175000 달러 → 17.5)

In [ ]:
houses = houses.assign(area_m2   = lambda x: (x['livingArea'] * 0.0929).round(1),
                       price_man = lambda x: x['price'] / 10000)

print(houses[['area_m2', 'price_man']].describe().round(1))

---

## 단계 4. 탐색

### 주장을 검토하는 방법
- 중개사의 설명을 하나의 **주장**으로 보고, 단계 4 ~ 6을 주장을 검토하는 데 맞게 나누어 진행함

| 할 일 | 내용 | 해당 단계 | 코드 |
|---|---|---|---|
| 주장의 숫자 직접 계산하기 | 주장에 나온 숫자가 데이터에서 어떻게 나왔는지 직접 계산함 | 단계 4 | 있음 |
| 다른 이유 찾기 | 주장이 틀릴 수 있는 이유(다른 요인 후보)를 적음 | 단계 4 | 없음 |
| 확인하기 | 후보 요인이 비슷한 집끼리 나누어 비교하고, 필요하면 검정함 | 단계 4, 5 | 있음 |
| 판단하기 | 주장을 받아들일지 정하고, 근거가 된 수치와 한계를 적음 | 단계 6 | 없음 |

- 판단은 네 가지 가운데 하나를 고름
  - **받아들임**: 다른 요인을 고려해도 주장한 만큼의 차이가 있음
  - **일부만 맞음**: 차이는 있지만 주장보다 작거나, 일부 경우에만 나타남
  - **받아들일 수 없음**: 다른 요인을 고려하면 차이가 거의 없거나 반대로 나타남
  - **판단할 수 없음**: 비교할 자료가 너무 적거나, 확인할 방법이 없음

- 예시 답을 맞히는 것보다 스스로 생각하고 판단하는 과정이 중요함
- 다른 이유로 꼽은 요인이 차이를 만들지 않은 것으로 확인되어도 의미 있는 결과임

### 주장의 숫자 직접 계산하기

> "벽난로가 있는 집은 **6.6만 달러** 더 비쌉니다."

- 중개사가 말한 이 숫자가 어떻게 나왔는지 데이터로 직접 계산해 봄

In [ ]:
houses = houses.assign(has_fireplace = lambda x: x['fireplaces'] > 0)   # 벽난로가 하나 이상이면 True

print(houses.groupby('has_fireplace')['price_man'].agg(['mean', 'size']).round(1))
# agg(['mean', 'size']) 의미 : mean과 size를 계산한 결과를 열로 나란히 붙임 (mean: 평균, size: 결측치 포함한 행의 수) (참고: count는 결측치 제외 값의 수)

- 벽난로가 없는 집은 평균 17.5만 달러, 있는 집은 24.1만 달러로 **6.6만 달러** 차이 남
- 중개사의 숫자는 이 평균 차이를 그대로 말한 것임

### 다른 이유 찾기

#### 판단해 볼 것
- 이 6.6만 달러가 모두 벽난로 값일까?
- 벽난로가 있는 집과 없는 집은 벽난로 말고 또 무엇이 다를 수 있을까? 후보를 두 가지 적고, 각각 이유를 한 줄로 쓸 것

> 내가 찾은 다른 이유:

<details>
<summary><b>생각의 근거 보기</b> (클릭)</summary>

- 벽난로는 보통 넓고 비싼 집에 설치되어 있을 가능성이 있음
  - 그렇다면 차이의 일부는 벽난로가 아니라 면적 때문일 수 있음
- 그 밖에도 집 나이, 침실 수, 욕실 수 등이 후보가 될 수 있음

</details>

### 확인하기

- 어떤 요인이 가격 차이를 만들었다면 그 요인은 두 가지 조건을 모두 만족해야 함
  - 조건 1: 벽난로가 있는 집과 없는 집에서 그 요인의 **값이 크게 다름**
  - 조건 2: 그 요인의 값이 비슷한 집끼리 비교하면, 벽난로가 있는 집과 없는 집의 가격 차이가 **크게 줄어듦**
- 두 조건을 모두 만족하면 11주차에서 다룬 교란 변수로 볼 수 있음
- 따라서 두 단계로 확인함

| 순서 | 확인할 것 | 사용할 표 |
|---|---|---|
| (1) 후보 좁히기 | 후보 요인의 값이 벽난로 유무에 따라 다른가 (조건 1) | 특성 평균 표 |
| (2) 나눠서 비교하기 | 후보 요인의 값이 비슷한 집끼리 비교하면 가격 차이가 크게 줄어드는가 (조건 2) | 표 A ~ C 가운데 필요한 표 |

#### (1) 후보 좁히기: 벽난로 유무별 집의 특성 평균

In [ ]:
features = houses.groupby('has_fireplace')[['area_m2', 'age', 'bathrooms', 'bedrooms', 'lotSize']].mean()
print(features.round(1))

##### 판단해 볼 것
- 나의 후보 가운데 벽난로 유무에 따라 크게 다른 것은 무엇인가?
- 벽난로 유무에 따라 거의 다르지 않은 요인은 무엇인가?

> 남은 후보:

<details>
<summary><b>후보를 좁히는 근거 보기</b> (클릭)</summary>

- 벽난로가 있는 집은 평균 50㎡ 더 넓고 (135 vs 185㎡), 12년 더 새것이며, 욕실이 0.5개, 침실이 0.4개 많음
  - 면적, 나이, 욕실 수, 침실 수는 조건 1을 만족하므로 후보로 남김
- 대지 면적은 두 집단 모두 0.5에이커로 같음
  - 두 집단에서 대지 면적이 비슷하므로, 벽난로 유무에 따른 가격 차이를 거의 설명하지 못함. 후보에서 제외함
- 조건 1만으로는 부족함. 그 요인이 실제로 가격 차이를 만드는지는 (2)에서 확인함

</details>

#### (2) 나눠서 비교하기
- 아래 표 A ~ C는 여러 기준으로 집을 나누어, 같은 구간 안에서 벽난로 유무에 따른 평균 가격을 비교한 것임
- (1)에서 남은 후보를 확인하는 데 **필요한 표를 골라** 판단에 사용함

##### 구간 나누기: `pd.qcut( )`
- 12주차에는 기온을 5도 간격으로 나눔 (`(temp // 5) * 5`). 구간의 **폭**이 같음
- 면적처럼 값이 한쪽에 몰린 변수를 같은 폭으로 나누면, 어떤 구간에는 집이 거의 없게 됨
- `pd.qcut(열, 4)` : 각 구간에 들어가는 **개수**가 비슷하도록 4개 구간으로 나눔
- `labels=`로 구간의 이름을 정할 수 있음

In [ ]:
houses = houses.assign(area_group = lambda x: pd.qcut(x['area_m2'], 4, labels=['1 (작음)', '2', '3', '4 (큼)']),
                       age_group  = lambda x: pd.qcut(x['age'], 4, labels=['1 (새 집)', '2', '3', '4 (오래됨)']))

# 구간의 경계와 구간마다 들어간 집의 수
print(pd.qcut(houses['area_m2'], 4).value_counts().sort_index())

print()
print(houses[['age', 'bedrooms', 'area_m2', 'price_man', 'has_fireplace', 'area_group', 'age_group']].head())

##### **표 A. 거주 면적 구간별 평균 가격 (만 달러)**

In [ ]:
table_a = houses.pivot_table(index='area_group', columns='has_fireplace', values='price_man', aggfunc='mean', observed=True)
count_a = houses.pivot_table(index='area_group', columns='has_fireplace', values='price_man', aggfunc='count', observed=True)

table_a['차이'] = table_a[True] - table_a[False]   # 구간마다 (벽난로 있음 - 없음)

print(table_a.round(1))
print("구간별 차이의 평균:", round(table_a['차이'].mean(), 1))
print()
print(count_a)   # 구간마다 집의 수

##### **표 B. 집 나이 구간별 평균 가격 (만 달러)**

In [ ]:
table_b = houses.pivot_table(index='age_group', columns='has_fireplace', values='price_man', aggfunc='mean', observed=True)
count_b = houses.pivot_table(index='age_group', columns='has_fireplace', values='price_man', aggfunc='count', observed=True)

table_b['차이'] = table_b[True] - table_b[False]

print(table_b.round(1))
print("구간별 차이의 평균:", round(table_b['차이'].mean(), 1))
print()
print(count_b) # 구간마다 집의 수

##### **표 C. 침실 수별 평균 가격 (만 달러)**

In [ ]:
# 침실 1개, 5개 이상은 집이 적어서 2 ~ 4개만 비교함
bed = houses.query('bedrooms in [2, 3, 4]')

table_c = bed.pivot_table(index='bedrooms', columns='has_fireplace', values='price_man', aggfunc='mean')
count_c = bed.pivot_table(index='bedrooms', columns='has_fireplace', values='price_man', aggfunc='count')

table_c['차이'] = table_c[True] - table_c[False]

print(table_c.round(1))
print("침실 수별 차이의 평균:", round(table_c['차이'].mean(), 1))
print()
print(count_c) # 구간마다 집의 수

##### 판단해 볼 것
- 표 A ~ C 가운데 나의 후보를 확인하는 데 필요한 표는 무엇인가? 그 표에서 차이는 어떻게 달라졌는가?
- 확인하고 싶은 후보에 맞는 표가 없다면, 어떤 표가 더 필요한가?

> 고른 표와 확인 결과:

<details>
<summary><b>표를 고르는 근거 보기</b> (클릭)</summary>

- **판단 기준**: 후보 요인이 차이를 만들었다면, 그 요인이 비슷한 집끼리 비교할 때 차이가 처음 차이(6.6만 달러)보다 **크게 줄어야** 함
- **표 A (면적)**
  - 구간별 차이가 0.5, 0.5, 2.5, 3.1만 달러로 다르지만, 평균하면 약 1.7만 달러로 처음 차이보다 크게 줄어듦
  - 벽난로가 없는 집만 보아도 면적 구간이 커질수록 14.2만 → 28.6만 달러로 가격이 뚜렷하게 오름. 면적은 가격과 강하게 관련됨
  - 확인 결과: **확인해 보니 해당되는 후보 (판단에 사용할 표)**
- **표 B (나이)**
  - 특성 평균 표에서 벽난로가 있는 집이 평균 12년 더 새것이었음 (조건 1 만족)
  - 그러나 나이 구간별 차이는 9.0, 6.2, 5.1, 4.4만 달러이고, 평균하면 약 6.2만 달러로 처음 차이(6.6만 달러)와 거의 같음 (조건 2 불만족)
  - 이유: 벽난로 유무와 관계없이 가장 새 집 구간만 가격이 높고 (없음 21.5, 있음 30.5만 달러), 나머지 세 구간은 비슷함 (없음 15.6 ~ 17.0, 있음 21.0 ~ 21.8만 달러). 집 나이에 따라 가격이 크게 달라지지 않음
  - 확인 결과: **확인해 보니 해당되지 않는 후보**
- **표 C (침실 수)**
  - 침실 수별 차이가 2.8, 4.9, 5.6만 달러이고, 평균하면 약 4.4만 달러로 처음 차이(6.6만 달러)보다 줄었지만 표 A만큼 크게 줄지는 않음
  - 확인 결과: **확인해 보니 일부만 해당되는 후보**
- **정리**

| | 면적 | 나이 | 침실 수 |
|---|---|---|---|
| 조건 1: 벽난로 유무에 따라 다른가 (특성 평균 표) | 다름 (50㎡) | 다름 (12년) | 다름 (0.4개) |
| 조건 2: 나누어 비교하면 차이가 크게 줄어드는가 (표 A ~ C) | 줄어듦 (평균 약 1.7만) | 거의 그대로 (평균 약 6.2만) | 조금 줄어듦 (평균 약 4.4만) |

- 욕실 수를 다른 이유로 꼽았다면 해당하는 표가 없음
  - 욕실 수가 비슷한 집끼리 나눈 표가 더 필요함. 스스로 표를 만들어서 비교해 볼 것

</details>

---

## 단계 5. 근거

### 확인하기 (계속): 면적 구간별 t 검정
- 표 A에서 구간 1, 2의 차이는 0.5만 달러, 구간 3, 4의 차이는 2.5 ~ 3.1만 달러였음
- 이 차이가 표본에 따라 우연히 생길 수 있는 정도인지 t 검정으로 확인함
  - 12주차와 같이 두 집단의 분산이 같다고 가정함
- 먼저 나누지 않은 전체 비교도 함께 검정해 봄

In [ ]:
from scipy import stats

# 전체 비교 (면적을 고려하지 않음)
yes = houses.query('has_fireplace == True')['price_man']
no  = houses.query('has_fireplace == False')['price_man']
print("전체      유의확률:", round(stats.ttest_ind(yes, no).pvalue, 4))
print()

# 면적 구간마다 비교
for g in ['1 (작음)', '2', '3', '4 (큼)']:
    part = houses.query('area_group == @g')
    yes = part.query('has_fireplace == True')['price_man']
    no  = part.query('has_fireplace == False')['price_man']
    print(g, " 유의확률:", round(stats.ttest_ind(yes, no).pvalue, 4))

<details>
<summary><b>내용 보기</b> (클릭)</summary>

- 전체 비교의 유의확률은 0에 가까움. 검정만 보면 "벽난로가 있으면 비싸다"가 확실한 것처럼 보임
- 면적 구간으로 나누면 결과가 달라짐
  - 구간 1, 2 (작은 집): 유의확률 0.38, 0.39로 0.05보다 큼. 벽난로 유무에 따른 차이가 **있다고 할 수 없음**
  - 구간 3, 4 (넓은 집): 유의확률 0.0002, 0.03으로 0.05보다 작음. 차이가 **있음**
- 검정은 차이가 우연인지를 알려 줄 뿐, 그 차이가 **무엇 때문인지는 알려 주지 않음**
  - 전체 비교의 아주 작은 유의확률에는 면적 차이까지 섞여 있기 때문에 교란 변수를 먼저 고려하고 나서 검정해야 의미 있는 근거가 됨

</details>

---

## 단계 6. 결론과 한계

### 판단하기

#### 판단해 볼 것
- 1번 설명을 받아들일 수 있는가? (받아들임 / 일부만 맞음 / 받아들일 수 없음 / 판단할 수 없음)
- 사려는 집의 크기에 따라 판단이 달라지는가?
- 판단의 근거가 된 수치를 함께 쓸 것

> 나의 판단과 근거:

<details>
<summary><b>판단의 근거 보기</b> (클릭)</summary>

- **일부만 맞음**
- 작은 집(구간 1, 2)에서는 벽난로 유무에 따른 차이가 있다고 할 수 없음 (차이 0.5만 달러, 유의확률 0.38, 0.39)
- 넓은 집(구간 3, 4)에서만 2.5 ~ 3.1만 달러 차이가 있음 (유의확률 0.0002, 0.03). 주장한 6.6만 달러의 절반에도 못 미침
- 벽난로가 있는 집이 비싼 이유의 상당 부분은 **집이 넓기 때문**임 (특성 평균 표, 표 A)
- 따라서 "벽난로가 있는 집은 6.6만 달러 더 비싸다"는 설명은 과장임
- 작은 집을 산다면 벽난로 때문에 더 낼 근거가 없고, 넓은 집이라면 2.5 ~ 3.1만 달러 정도까지가 시장 가격에 맞는 범위임

</details>

### 더 생각해 볼 것: 한계
- 면적 구간 3, 4에서 남은 2.5 ~ 3.1만 달러는 벽난로 덕분일까?
  - 넓은 집끼리 비교해도, 벽난로가 있는 집이 침실이나 욕실이 더 많을 수 있음 (데이터에 있지만 고려하지 않은 요인)
  - 한 가지 요인으로 나누어 본 것만으로는 다른 요인의 영향을 모두 없앨 수 없음
- 데이터에 없는 특징의 영향은 확인할 수 없음
  - 예: 벽난로가 있는 집은 마감이 고급스럽거나 관리가 잘 되어 있을 수 있지만, 이런 정보는 데이터에 없음
  - 따라서 남은 차이가 벽난로 자체의 값인지는 구분할 수 없음
- 2006년 한 지역의 거래이므로, 지금 다른 지역의 집값에 그대로 적용할 수는 없음
- 이런 내용은 결론이 아니라 **한계**에 적어야 함

---

### 지금까지의 판단 정리

| 단계 | 판단 | 이유 |
|---|---|---|
| 3. 정제 | 모든 값이 같은 7행을 하나씩만 남김 | 같은 기록이 두 번 입력된 것으로 보임 |
| 3. 정제 | 가격이 토지 가치보다 낮은 3채 제외 | 정상적인 시장 거래로 보기 어려움 |
| 3. 정제 | 제곱미터, 만 달러로 변환 | 읽고 비교하기 쉽게 하기 위함 |
| 4. 탐색 (주장의 숫자 직접 계산하기) | 6.6만 달러는 벽난로 유무별 평균 가격의 차이임 | 주장의 숫자가 어떻게 나왔는지 확인 |
| 4. 탐색 (다른 이유 찾기) | 면적, 나이, 침실 수, 욕실 수 등을 후보로 떠올림 | 벽난로가 있는 집은 다른 점도 많을 수 있음 |
| 4. 탐색 (확인하기) | 대지 면적은 후보에서 제외. 면적 구간별로 나누면 차이가 크게 줄고, 나이 구간별로는 거의 그대로, 침실 수별로는 조금 줄어듦 | 후보 요인이 비슷한 집끼리 비교하기 위함 |
| 5. 근거 (확인하기) | 작은 집은 차이가 있다고 할 수 없고, 넓은 집만 차이가 있음 | 면적 구간별 t 검정 |
| 6. 결론과 한계 (판단하기) | 일부만 맞음 | 면적이 비슷한 집끼리는 차이가 없거나 2.5 ~ 3.1만 달러로 작음 |

### 가족에게 보내는 조언 (예시)

| 구분 | 내용 |
|---|---|
| 결론 (사실) | 벽난로가 있는 집은 없는 집보다 평균 6.6만 달러 비쌌음 (17.5만 vs 24.1만 달러). 그러나 벽난로가 있는 집은 평균 50㎡ 더 넓었고 (특성 평균 표), 거주 면적이 비슷한 집끼리 비교하면 작은 집에서는 차이가 있다고 할 수 없었고 (유의확률 0.38, 0.39), 넓은 집에서만 2.5 ~ 3.1만 달러 차이가 있었음 (표 A, 유의확률 0.0002, 0.03) |
| 구매 판단 | 작은 집을 산다면 벽난로 때문에 더 낼 근거가 없다고 볼 수 있음. 넓은 집이라면 벽난로가 없는 비슷한 크기의 집보다 2.5 ~ 3.1만 달러 정도 비싼 것까지는 시장 가격에 맞는 범위로 볼 수 있음 |
| 한계 | 남은 차이가 벽난로 자체의 값인지, 벽난로가 있는 집의 다른 특징(마감, 관리 상태 등) 때문인지는 구분할 수 없음. 면적 외의 요인(침실 수, 욕실 수 등)의 영향도 남아 있음. 2006년 한 지역의 거래만 사용함 |

- 결론에는 처음 보인 차이와 **다른 요인을 고려한 뒤의 차이**를 함께 씀
- 구매 판단은 데이터로 확인한 범위까지만 씀. 관계를 원인처럼 쓰지 않도록 주의함 (예: "벽난로를 달면 집값이 오름"처럼 쓰지 않음)
- 한계에는 이 데이터로 **확인할 수 없는 것**을 구체적으로 씀

### 실습에서 이어서 할 것

| 문제 | 검토할 내용 |
|---|---|
| 1 | 2번 설명: "중앙냉방이 있는 집은 6.8만 달러 더 비쌉니다." |
| 2 | 3번 설명: "침실이 4개인 집은 3개인 집보다 6.5만 달러 더 비쌉니다." |

- 데이터 준비(단계 2, 3)는 이번에 수행한 것과 같은 판단을 적용한 상태로 시작함
- 모든 문제를 단계 4 ~ 6의 **주장의 숫자 직접 계산하기 → 다른 이유 찾기 → 확인하기 → 판단하기** 순서로 진행함